# 11/06 판다스 데이터분석 과제

## 준비 셀 1 — import

In [ ]:
import pandas as pd
import seaborn as sns

## 준비 셀 2 — 데이터 다운로드

처음 한 번만 실행하세요. 실행 전에 `data/raw` 폴더를 만들어 두세요.

In [ ]:
titanic_raw = sns.load_dataset("titanic")
titanic_raw.to_csv("data/raw/titanic.csv", index=False)
print("저장 완료")

## 준비 셀 3 — 데이터 불러오기

In [ ]:
titanic = pd.read_csv("data/raw/titanic.csv")
print(titanic.shape)
print(titanic.head(3))

## 문제 1. 열 선택

In [ ]:
# 1-1. fare 열 하나 선택
fare_col = titanic["fare"]
print(type(fare_col))

# 1-2. 4개 열 선택
selected_cols = titanic[["survived", "pclass", "sex", "age"]]
print(type(selected_cols))
print(selected_cols.shape)

# 1-3. 주석: 대괄호 한 겹으로 열 하나를 선택하면 Series이고,
#      열 이름을 리스트로 묶어 대괄호 두 겹으로 선택하면 DataFrame이다.

## 문제 2. loc vs iloc

In [ ]:
# 2-1. loc은 인덱스 이름표 기준이며 끝을 포함한다.
loc_result = titanic.loc[10:15, ["sex", "age"]]
print(loc_result)

# 2-2. iloc은 위치 번호 기준이며 끝을 포함하지 않는다.
iloc_result = titanic.iloc[10:15, [2, 3]]
print(iloc_result)

# 2-3. 행 수 확인
print("loc 행 수:", loc_result.shape[0])
print("iloc 행 수:", iloc_result.shape[0])

# 주석: loc의 10:15는 인덱스 10부터 15까지 포함하여 6행이고,
#      iloc의 10:15는 위치 10부터 14까지 선택하여 5행이다.

## 문제 3. iloc 응용

In [ ]:
# 마지막 3행, 앞의 4개 열
print(titanic.iloc[-3:, :4])

## 문제 4. 조건 마스크

In [ ]:
# 4-1. 10살 미만인지 검사하는 마스크
under_10 = titanic["age"] < 10
print(under_10.head(5))

# 4-2. 10살 미만 승객 수
print("10살 미만 승객 수:", titanic[under_10].shape[0])

## 문제 5. 복합 조건

In [ ]:
# 5-1. 3등석 그리고 남성 그리고 생존
mask1 = (titanic["pclass"] == 3) & (titanic["sex"] == "male") & (titanic["survived"] == 1)
print("조건을 만족하는 승객 수:", titanic[mask1].shape[0])

# 5-2. 1등석 또는 요금 100 이상
mask2 = (titanic["pclass"] == 1) | (titanic["fare"] >= 100)
print("조건을 만족하는 승객 수:", titanic[mask2].shape[0])

# 5-3. 혼자 탑승하지 않은 승객
mask3 = ~titanic["alone"]
print("조건을 만족하는 승객 수:", titanic[mask3].shape[0])

## 문제 6. isin · between

In [ ]:
# 6-1. Cherbourg 또는 Queenstown 탑승
mask_port = titanic["embark_town"].isin(["Cherbourg", "Queenstown"])
print(titanic[mask_port].shape)

# 6-2. 나이 20~30세 (양 끝 포함)
mask_age = titanic["age"].between(20, 30)
age_group = titanic[mask_age]
print(age_group.shape)
print("나이 최솟값:", age_group["age"].min())
print("나이 최댓값:", age_group["age"].max())

# 6-3. 양 끝 미포함
mask_age_neither = titanic["age"].between(20, 30, inclusive="neither")
print(titanic[mask_age_neither].shape)

## 문제 7. loc[조건, 열]

In [ ]:
# 12살 이하 그리고 3등석
kids_mask = (titanic["age"] <= 12) & (titanic["pclass"] == 3)
kids = titanic.loc[kids_mask, ["age", "sex", "survived"]]

print("행 수:", kids.shape[0])
print(f"생존률: {kids['survived'].mean():.2f}")

## 문제 8. query

In [ ]:
# 8-1. Cherbourg 탑승 그리고 요금 50 이상
min_fare = 50
query_result = titanic.query("embark_town == 'Cherbourg' and fare >= @min_fare")
print(query_result.shape)

# 8-2. 문제 5-1 조건을 query로 작성
query_result_2 = titanic.query("pclass == 3 and sex == 'male' and survived == 1")
print("query 행 수:", query_result_2.shape[0])
print("마스크 행 수:", titanic[mask1].shape[0])

## 문제 9. 정렬

In [ ]:
# 9-1. 객실 등급 오름차순, 같은 등급 안에서는 요금 내림차순
sorted_titanic = titanic.sort_values(
    by=["pclass", "fare"],
    ascending=[True, False]
)
print(sorted_titanic[["pclass", "fare", "sex", "age"]].head(3))

# 9-2. 요금 내림차순 정렬
fare_sorted = titanic.sort_values(by="fare", ascending=False)
print("iloc[0]의 fare:", fare_sorted.iloc[0]["fare"])
print("loc[0]의 fare:", fare_sorted.loc[0]["fare"])

# 주석: iloc[0]은 정렬된 결과의 첫 번째 행 위치이고,
#      loc[0]은 정렬 후에도 유지된 인덱스 이름표가 0인 행이다.

## 문제 10. 종합 — 20~40세 남성, 1·2등석 vs 3등석

In [ ]:
# 10-1. 남성, 나이 20~40세, 1·2등석
male_age_mask = (titanic["sex"] == "male") & (titanic["age"].between(20, 40)) & (titanic["pclass"].isin([1, 2]))
male_12 = titanic.loc[male_age_mask, ["pclass", "age", "fare", "survived"]]

# 10-2. 행 수, 생존률, 요금 상위 3명
print("행 수:", male_12.shape[0])
survival_12 = male_12["survived"].mean()
print(f"1·2등석 생존률: {survival_12:.2f}")
print("요금 상위 3명:")
print(male_12.sort_values(by="fare", ascending=False).head(3))

# 10-3. 같은 조건을 query로 검증
male_12_query = titanic.query("sex == 'male' and age >= 20 and age <= 40 and pclass in [1, 2]")
print("query 행 수:", male_12_query.shape[0])

# 10-4. 등급을 3등석으로 변경
male_3_mask = (titanic["sex"] == "male") & (titanic["age"].between(20, 40)) & (titanic["pclass"] == 3)
male_3 = titanic.loc[male_3_mask, ["pclass", "age", "fare", "survived"]]
survival_3 = male_3["survived"].mean()

print("3등석 행 수:", male_3.shape[0])
print(f"3등석 생존률: {survival_3:.2f}")

### 문제 10 해석

20~40세 남성 승객 중 1·2등석의 생존률은 위 결과에서 확인할 수 있다. 3등석의 생존률과 비교해 어느 등급의 생존률이 더 높은지, 두 생존률의 차이가 얼마나 되는지 결과를 바탕으로 2~3문장으로 작성한다.